In [25]:
import pandas as pd  # 데이터 처리
from pathlib import Path  # 경로 처리
import math  # 올림 계산

# =========================
# 0) 파일 경로
# =========================
SRC_XLSX = Path("상품DB_251218.xlsx")  # 원본 엑셀 경로
CAT_XLSX = Path("피프틴양식_작성중.xlsx")  # 카테고리 마스터 엑셀 경로  # 추가
OUT_7TABLES = Path("피프틴양식_ver2.xlsx")  # 7개 테이블 출력
OUT_MERGED = Path("상품통합_ver2.xlsx")  # 통합DB 출력

# =========================
# 1) 고정 마스터 (language)
# =========================
LANGUAGE_ROWS = [  # 언어 마스터 고정값
    ("01", "한국어", "ko"),  # 한국어
    ("02", "영어", "en"),  # 영어
    ("03", "태국어", "th"),  # 태국어
    ("04", "베트남어", "vi"),  # 베트남어
    ("05", "인도네시아어", "id"),  # 인도네시아어
]  # 언어 목록 끝

# =========================
# 2) 카테고리 매핑 (KO 기준)
# =========================

UNIT_CODE = {  # 옵션단위 코드
    "EA": "01",  # 낱개
    "PACK": "02",  # 팩
    "BUNDLE": "03",  # 번들
    "BOX": "04",  # 박스
}  # 옵션단위 끝

# =========================
# 3) 유틸 함수
# =========================
def zfill_num(s: pd.Series, n: int) -> pd.Series:  # 0채우기
    return s.astype("Int64").astype("string").str.zfill(n)  # 숫자->문자->zfill

def safe_col(df: pd.DataFrame, name: str) -> pd.Series:  # 컬럼 없으면 빈값
    return df[name] if name in df.columns else pd.Series([pd.NA] * len(df))  # 안전 반환

def rename_cols(df: pd.DataFrame, col_map: dict) -> pd.DataFrame:  # 컬럼명 바꾸기
    return df.rename(columns=col_map)  # 컬럼명 변경

# =========================
# 4) 원본 시트 읽기
# =========================
base = pd.read_excel(SRC_XLSX, sheet_name=0)  # 1시트(기본)
price = pd.read_excel(SRC_XLSX, sheet_name=1)  # 2시트(가격/다국어)

# =========================
# 4-1) 카테고리 마스터 읽기(상위/하위)  # 추가
# =========================
top_master = pd.read_excel(CAT_XLSX, sheet_name=1, dtype="string")  # 2번째 시트(상위)  # 추가
sub_master = pd.read_excel(CAT_XLSX, sheet_name=2, dtype="string")  # 3번째 시트(하위)  # 추가

top_master = top_master.rename(columns={  # 상위 컬럼명 통일  # 추가
    "카테고리 No (PK)": "top_category_no",  # PK  # 추가
    "언어 No (FK)": "language_no",  # 언어  # 추가
    "카테고리명": "category_name",  # 이름  # 추가
})  # 끝  # 추가

sub_master = sub_master.rename(columns={  # 하위 컬럼명 통일  # 추가
    "하위 카테고리 No (PK)": "sub_category_no",  # PK  # 추가
    "상위 카테고리 No (FK)": "top_category_no",  # 상위FK  # 추가
    "언어 No (FK)": "language_no",  # 언어  # 추가
    "하위 카테고리명": "sub_category_name",  # 이름  # 추가
})  # 끝  # 추가

top_master["top_category_no"] = top_master["top_category_no"].astype("string").str.zfill(4)  # 4자리  # 추가
top_master["language_no"] = top_master["language_no"].astype("string").str.zfill(2)  # 2자리  # 추가
sub_master["sub_category_no"] = sub_master["sub_category_no"].astype("string").str.zfill(7)  # 7자리  # 추가
sub_master["top_category_no"] = sub_master["top_category_no"].astype("string").str.zfill(4)  # 4자리  # 추가
sub_master["language_no"] = sub_master["language_no"].astype("string").str.zfill(2)  # 2자리  # 추가

# =========================
# 4-2) KO 하위카테고리명 -> 번호 매핑 만들기  # 추가
# =========================
sub_map_ko = (  # 매핑DF  # 추가
    sub_master[sub_master["language_no"] == "01"]  # 한국어만  # 추가
    .set_index("sub_category_name")[["sub_category_no", "top_category_no"]]  # 이름키  # 추가
    .to_dict("index")  # dict  # 추가
)  # 끝  # 추가

base["품목코드"] = base["품목코드"].astype("Int64")  # 품목코드 정리
price["품목코드"] = price["품목코드"].astype("Int64")  # 품목코드 정리

base["product_no"] = zfill_num(base["품목코드"], 8)  # 상품No(8자리)
price["product_no"] = zfill_num(price["품목코드"], 8)  # 상품No(8자리)

# =========================
# 5) “처음부터 상품명 붙여서” 조합용 기준 DF 만들기
# =========================
base_cols = ["product_no"]  # 기본 컬럼
for col in ["품목명", "국가명", "과세/면세", "카테고리명", "단위", "규격정보"]:  # 필요한 컬럼들
    if col in base.columns:  # 있으면 추가
        base_cols.append(col)  # 컬럼 추가

df = price.merge(  # 가격시트에
    base[base_cols].copy(),  # 기본정보 붙이기
    on="product_no",  # 상품No로 조인
    how="left",  # 왼쪽 유지
)  # merge 끝

df["origin_product_name"] = safe_col(df, "품목명_x")  # 원본 상품명

# =========================
# 6) 카테고리 번호 생성  # 수정(마스터 참조)
# =========================
df["sub_name_ko"] = safe_col(df, "카테고리").fillna(df["카테고리명"])  # 하위카테고리명 후보  # 유지
df["sub_category_no"] = df["sub_name_ko"].map(lambda x: sub_map_ko.get(x, {}).get("sub_category_no"))  # 하위번호  # 추가
df["top_category_no"] = df["sub_name_ko"].map(lambda x: sub_map_ko.get(x, {}).get("top_category_no"))  # 상위번호  # 추가

# =========================
# 7) 7개 테이블 생성
# =========================

language = pd.DataFrame(LANGUAGE_ROWS, columns=["language_no", "language_name", "language_code"])  # 언어테이블

top_category = (  # 상위카테고리  # 수정
    top_master[["top_category_no", "language_no", "category_name"]]  # 마스터에서 가져오기  # 수정
    .dropna()  # 빈값 제거  # 수정
    .drop_duplicates()  # 중복 제거  # 수정
)  # 끝  # 수정

sub_category = (  # 하위카테고리  # 수정
    sub_master[["sub_category_no", "top_category_no", "language_no", "sub_category_name"]]  # 마스터에서 가져오기  # 수정
    .dropna()  # 빈값 제거  # 수정
    .drop_duplicates()  # 중복 제거  # 수정
)  # 끝  # 수정

product = pd.DataFrame()  # product 만들기
product["product_no"] = df["product_no"]  # 상품No
product["top_category_no"] = df["top_category_no"]  # 상위FK(4자리)
product["sub_category_no"] = df["sub_category_no"]  # 하위FK(7자리)
product["one_time_limit_yn"] = 0  # 기본 0
product["tax_free_yn"] = (df["과세/면세"] == "면세").astype("int")  # 면세면 1
product["wms_code"] = zfill_num(df["품목코드"], 8)  # WMS코드
product["item_code"] = zfill_num(df["품목코드"], 8)  # 품목코드
product["coupang_proxy_yn"] = 0  # 기본 0
product["safety_stock"] = pd.NA  # 안전재고
product["temp_soldout_yn"] = 0  # 임시품절
product["timesale_start_dt"] = pd.NA  # 타임세일 시작
product["timesale_end_dt"] = pd.NA  # 타임세일 종료
product["timesale_discount_type"] = pd.NA  # 할인유형
product["timesale_discount_value"] = pd.NA  # 할인값
product = product.drop_duplicates(subset=["product_no"])  # 중복 제거

def make_product_lang(lang_no: str, name_col: str) -> pd.DataFrame:  # 언어별 상품정보
    t = pd.DataFrame()  # 임시 DF
    t["product_no"] = df["product_no"]  # FK
    t["language_no"] = lang_no  # 언어 FK
    if name_col == "품목명":  # 원본명
        t["product_name"] = df["origin_product_name"]  # 한글명
    else:  # 다국어명
        t["product_name"] = safe_col(df, name_col)  # 해당명
    t["origin"] = safe_col(df, "국가명")  # 원산지
    t["description"] = pd.NA  # 상세설명
    t["notice"] = pd.NA  # 고시
    t = t[t["product_name"].notna()]  # 이름없는 행 제거
    t["product_lang_no"] = t["product_no"] + t["language_no"]  # PK
    return t[["product_lang_no", "product_no", "language_no", "product_name", "origin", "description", "notice"]]  # 반환

pl_ko = make_product_lang("01", "품목명")  # 한국
pl_en = make_product_lang("02", "상품 영어명")  # 영어
pl_th = make_product_lang("03", "태국")  # 태국
pl_vi = make_product_lang("04", "베트남")  # 베트남
pl_id = make_product_lang("05", "인도네시아")  # 인도네시아
product_lang = pd.concat([pl_ko, pl_en, pl_th, pl_vi, pl_id], ignore_index=True)  # 합치기

# 7-6) product_option
opt_rows = []  # 옵션 행
for _, r in df.iterrows():  # 한줄씩
    pno = r["product_no"]  # 상품No
    base_unit = str(r["단위"]) if pd.notna(r.get("단위")) else "EA"  # 기본단위
    base_code = UNIT_CODE.get(base_unit, "01")  # 코드
    opt_rows.append((pno + base_code, pno, base_code, r.get("규격정보"), 1, pd.NA, pd.NA))  # 기본옵션
    if pd.notna(r.get("팩품목코드")):  # 팩
        opt_rows.append((pno + "02", pno, "02", r.get("규격정보"), r.get("팩품목환산수량\n(1팩에 몇 개)"), pd.NA, pd.NA))  # 팩옵션
    if pd.notna(r.get("번들품목코드")):  # 번들
        opt_rows.append((pno + "03", pno, "03", r.get("규격정보"), r.get("번들품목환산수량\n(번들에 몇 개)"), pd.NA, pd.NA))  # 번들옵션
    if pd.notna(r.get("박스품목 환산수량")):  # 박스
        opt_rows.append((pno + "04", pno, "04", r.get("규격정보"), r.get("박스품목 환산수량"), pd.NA, pd.NA))  # 박스옵션

product_option = pd.DataFrame(  # 옵션 DF
    opt_rows,  # 데이터
    columns=["product_option_no", "product_no", "option_unit", "weight", "sale_unit", "max_buy_qty", "temp_soldout_text"],  # 컬럼
)
product_option = product_option.drop_duplicates(subset=["product_option_no"])  # 중복 제거

# =========================
# 7-6-1) 최대 구매 수량 자동 계산(바로 위 단위 기준)
# =========================
product_option["option_unit"] = product_option["option_unit"].astype("string").str.zfill(2)  # 단위코드 정리
product_option["sale_unit"] = pd.to_numeric(product_option["sale_unit"], errors="coerce")  # 판매단위 숫자
def _calc_max_buy_qty(g: pd.DataFrame) -> pd.DataFrame:  # 계산
    g = g.copy()  # 복사
    g["unit_int"] = pd.to_numeric(g["option_unit"], errors="coerce")  # 숫자화
    g["max_buy_qty"] = pd.NA  # 초기화
    units = sorted([u for u in g["unit_int"].dropna().unique().tolist()])  # 정렬
    next_map = {}  # 매핑
    for u in units:  # 반복
        higher = [x for x in units if x > u]  # 상위 후보
        next_map[u] = min(higher) if higher else None  # 바로 위
    for u in units:  # 반복
        nu = next_map.get(u)  # 위 단위
        if nu is None:  # 없으면
            continue  # 종료
        cur_sale = g.loc[g["unit_int"] == u, "sale_unit"]  # 현재
        next_sale = g.loc[g["unit_int"] == nu, "sale_unit"]  # 위
        if cur_sale.empty or next_sale.empty:  # 없으면
            continue  # 종료
        cur_val = cur_sale.iloc[0]  # 값
        next_val = next_sale.iloc[0]  # 값
        if pd.isna(cur_val) or pd.isna(next_val) or cur_val == 0:  # 예외
            continue  # 종료
        qty = int(math.ceil(float(next_val) / float(cur_val)))  # 올림
        g.loc[g["unit_int"] == u, "max_buy_qty"] = qty  # 저장
    g = g.drop(columns=["unit_int"])  # 정리
    return g  # 반환
product_option = product_option.groupby("product_no", group_keys=False).apply(_calc_max_buy_qty)  # 적용

# =========================
# 7-7) product_option_lang_price (언어별 가격 “전부”)
#    - 여기서 "입고가/원가/판매가 * 판매단위" 적용  # 수정
# =========================
def add_price_row(rows: list, opt_no: str, lang_no: str, inbound, cost, sale, sale_unit):  # 가격행 추가  # 수정
    rows.append((opt_no + lang_no, lang_no, opt_no, inbound, cost, sale, sale_unit))  # 판매단위 같이 저장  # 수정

price_rows = []  # 가격 행
for _, r in df.iterrows():  # 반복
    pno = r["product_no"]  # 상품No
    base_unit = str(r["단위"]) if pd.notna(r.get("단위")) else "EA"  # 기본단위
    base_code = UNIT_CODE.get(base_unit, "01")  # 코드
    ea_no = pno + base_code  # 옵션No

    ea_sale_unit = 1  # EA 판매단위  # 기본
    for lang_no in ["01", "02", "03", "04", "05"]:  # 언어 반복
        add_price_row(price_rows, ea_no, lang_no, r.get("입고단가"), r.get("낱개단가"), r.get("낱개단가"), ea_sale_unit)  # 추가

    if pd.notna(r.get("팩품목코드")):  # 팩이면
        pack_no = pno + "02"  # 팩 옵션No
        pack_sale_unit = r.get("팩품목환산수량\n(1팩에 몇 개)")  # 팩 판매단위
        for lang_no in ["01", "02", "03", "04", "05"]:  # 언어 반복
            add_price_row(price_rows, pack_no, lang_no, r.get("입고단가"), r.get("팩단가"), r.get("팩단가"), pack_sale_unit)  # 추가

    if pd.notna(r.get("번들품목코드")):  # 번들이면
        bun_no = pno + "03"  # 번들 옵션No
        bun_sale_unit = r.get("번들품목환산수량\n(번들에 몇 개)")  # 번들 판매단위
        for lang_no in ["01", "02", "03", "04", "05"]:  # 언어 반복
            add_price_row(price_rows, bun_no, lang_no, r.get("입고단가"), r.get("번들단가"), r.get("번들단가"), bun_sale_unit)  # 추가

    if pd.notna(r.get("박스품목 환산수량")):  # 박스면
        box_no = pno + "04"  # 박스 옵션No
        box_sale_unit = r.get("박스품목 환산수량")  # 박스 판매단위
        for lang_no in ["01", "02", "03", "04", "05"]:  # 언어 반복
            add_price_row(price_rows, box_no, lang_no, r.get("입고단가"), r.get("박스단가"), r.get("박스단가"), box_sale_unit)  # 추가

product_option_lang_price = pd.DataFrame(  # 가격 DF
    price_rows,  # 데이터
    columns=["option_lang_no", "language_no", "product_option_no", "inbound_price", "cost_price", "sale_price", "sale_unit"],  # 판매단위 포함  # 수정
)  # DF 생성

product_option_lang_price["sale_unit"] = pd.to_numeric(product_option_lang_price["sale_unit"], errors="coerce").fillna(1)  # 판매단위 정리  # 수정
product_option_lang_price["inbound_price"] = pd.to_numeric(product_option_lang_price["inbound_price"], errors="coerce")  # 숫자화  # 수정
product_option_lang_price["cost_price"] = pd.to_numeric(product_option_lang_price["cost_price"], errors="coerce")  # 숫자화  # 수정
product_option_lang_price["sale_price"] = pd.to_numeric(product_option_lang_price["sale_price"], errors="coerce")  # 숫자화  # 수정

product_option_lang_price["inbound_price"] = product_option_lang_price["inbound_price"] * product_option_lang_price["sale_unit"]  # 입고가*판매단위  # 수정
product_option_lang_price["cost_price"] = product_option_lang_price["cost_price"] * product_option_lang_price["sale_unit"]  # 원가*판매단위  # 수정
product_option_lang_price["sale_price"] = product_option_lang_price["sale_price"] * product_option_lang_price["sale_unit"]  # 판매가*판매단위  # 수정

product_option_lang_price = product_option_lang_price.drop(columns=["sale_unit"])  # 임시 판매단위 제거  # 수정
product_option_lang_price = product_option_lang_price.drop_duplicates(subset=["product_option_no", "language_no"])  # 중복 제거

# =========================
# 8) 통합DB 생성
# =========================
merged = product_option_lang_price.merge(product_option, on="product_option_no", how="left")  # 가격+옵션
merged = merged.merge(product, on="product_no", how="left")  # +상품
merged = merged.merge(product_lang, on=["product_no", "language_no"], how="left")  # +언어별 상품
merged = merged.merge(df[["product_no", "origin_product_name"]].drop_duplicates(), on="product_no", how="left")  # +원본명

# =========================
# 8-1) 컬럼명 한글로 변경(여기서 적용)
# =========================
language = rename_cols(language, {  # language 컬럼명 변경
    "language_no": "언어 No",  # 언어 PK
    "language_name": "언어명",  # 언어명
    "language_code": "언어 코드",  # 언어코드
})  # language 끝

top_category = rename_cols(top_category, {  # top_category 컬럼명 변경
    "top_category_no": "카테고리 No (PK)",  # 카테고리 PK
    "language_no": "언어 No (FK)",  # 언어 FK
    "category_name": "카테고리명",  # 카테고리명
})  # top_category 끝

sub_category = rename_cols(sub_category, {  # sub_category 컬럼명 변경
    "sub_category_no": "하위 카테고리 No (PK)",  # 하위 PK
    "top_category_no": "상위 카테고리 No (FK)",  # 상위 FK
    "language_no": "언어 No (FK)",  # 언어 FK
    "sub_category_name": "하위 카테고리명",  # 하위명
})  # sub_category 끝

product = rename_cols(product, {  # product 컬럼명 변경
    "product_no": "상품 No (PK)",  # 상품 PK
    "top_category_no": "상위 카테고리 No (FK)",  # 상위 FK
    "sub_category_no": "하위 카테고리 No (FK)",  # 하위 FK
    "one_time_limit_yn": "1회 구매 제한 상품 여부 (0-X, 1-O)",  # 제한 여부
    "tax_free_yn": "면세 상품 여부 (0-X, 1-O)",  # 면세 여부
    "wms_code": "WMS 코드",  # WMS 코드
    "item_code": "품목 코드",  # 품목 코드
    "coupang_proxy_yn": "쿠팡 대리구매 상품 여부 (0-X, 1-O)",  # 쿠팡 여부
    "safety_stock": "안전재고",  # 안전재고
    "temp_soldout_yn": "임시품절 여부 (0-X, 1-O)",  # 임시품절
    "timesale_start_dt": "타임세일 시작일시",  # 타임세일 시작
    "timesale_end_dt": "타임세일 종료일시",  # 타임세일 종료
    "timesale_discount_type": "타임세일 할인 유형 (percent => 정률, price => 정액)",  # 할인유형
    "timesale_discount_value": "타임세일 할인값",  # 할인값
})  # product 끝

product_lang = rename_cols(product_lang, {  # product_lang 컬럼명 변경
    "product_lang_no": "언어별 상품정보 No (PK)",  # 언어별 PK
    "product_no": "상품 No (FK)",  # 상품 FK
    "language_no": "언어 No (FK)",  # 언어 FK
    "product_name": "상품명",  # 상품명
    "origin": "원산지",  # 원산지
    "description": "상세 설명",  # 상세설명
    "notice": "상품 정보고시",  # 고시
})  # product_lang 끝

product_option = rename_cols(product_option, {  # product_option 컬럼명 변경
    "product_option_no": "상품 옵션 No (PK)",  # 옵션 PK
    "product_no": "상품 No (FK)",  # 상품 FK
    "option_unit": "옵션 단위(EA, BUNDLE, PACK, BOX)",  # 옵션단위
    "weight": "무게",  # 무게
    "sale_unit": "판매 단위",  # 판매단위
    "max_buy_qty": "최대 구매 수량",  # 최대수량
    "temp_soldout_text": "임시 품절여부",  # 임시품절 텍스트
})  # product_option 끝

product_option_lang_price = rename_cols(product_option_lang_price, {  # 가격 컬럼명 변경
    "option_lang_no": "언어별 상품 옵션정보 No (PK)",  # PK
    "language_no": "언어 No (FK)",  # 언어 FK
    "product_option_no": "상품 옵션 No (FK)",  # 옵션 FK
    "inbound_price": "입고가",  # 입고가
    "cost_price": "원가",  # 원가
    "sale_price": "판매가",  # 판매가
})  # 가격 끝

merged = rename_cols(merged, {  # merged 컬럼명 변경
    "origin_product_name": "원본 상품명",  # 원본명
    "option_lang_no": "언어별 상품 옵션정보 No (PK)",  # PK
    "language_no": "언어 No (FK)",  # 언어 FK
    "product_option_no": "상품 옵션 No (FK)",  # 옵션 FK
    "product_no": "상품 No (FK)",  # 상품 FK
    "inbound_price": "입고가",  # 입고가
    "cost_price": "원가",  # 원가
    "sale_price": "판매가",  # 판매가
    "option_unit": "옵션 단위(EA, BUNDLE, PACK, BOX)",  # 옵션단위
    "weight": "무게",  # 무게
    "sale_unit": "판매 단위",  # 판매단위
    "max_buy_qty": "최대 구매 수량",  # 최대수량
    "temp_soldout_text": "임시 품절여부",  # 임시품절 텍스트
    "product_name": "상품명",  # 상품명
    "origin": "원산지",  # 원산지
    "description": "상세 설명",  # 상세설명
    "notice": "상품 정보고시",  # 고시
    "top_category_no": "상위 카테고리 No (FK)",  # 상위 FK
    "sub_category_no": "하위 카테고리 No (FK)",  # 하위 FK
    "one_time_limit_yn": "1회 구매 제한 상품 여부 (0-X, 1-O)",  # 제한 여부
    "tax_free_yn": "면세 상품 여부 (0-X, 1-O)",  # 면세 여부
    "wms_code": "WMS 코드",  # WMS 코드
    "item_code": "품목 코드",  # 품목 코드
    "coupang_proxy_yn": "쿠팡 대리구매 상품 여부 (0-X, 1-O)",  # 쿠팡 여부
    "safety_stock": "안전재고",  # 안전재고
    "temp_soldout_yn": "임시품절 여부 (0-X, 1-O)",  # 임시품절
    "timesale_start_dt": "타임세일 시작일시",  # 타임세일 시작
    "timesale_end_dt": "타임세일 종료일시",  # 타임세일 종료
    "timesale_discount_type": "타임세일 할인 유형 (percent => 정률, price => 정액)",  # 할인유형
    "timesale_discount_value": "타임세일 할인값",  # 할인값
})  # merged 끝

# =========================
# 9) 엑셀 저장 (7테이블 / 통합DB)
# =========================
with pd.ExcelWriter(OUT_7TABLES, engine="openpyxl") as w:  # 7테이블 저장
    language.to_excel(w, sheet_name="language", index=False)  # language
    top_category.to_excel(w, sheet_name="top_category", index=False)  # top_category
    sub_category.to_excel(w, sheet_name="sub_category", index=False)  # sub_category
    product.to_excel(w, sheet_name="product", index=False)  # product
    product_lang.to_excel(w, sheet_name="product_lang", index=False)  # product_lang
    product_option.to_excel(w, sheet_name="product_option", index=False)  # product_option
    product_option_lang_price.to_excel(w, sheet_name="product_option_lang_price", index=False)  # price

with pd.ExcelWriter(OUT_MERGED, engine="openpyxl") as w:  # 통합 저장
    merged.to_excel(w, sheet_name="merged_all", index=False)  # merged

print("완료:", OUT_7TABLES, OUT_MERGED)  # 출력


C:\Users\jusun\AppData\Local\Temp\ipykernel_5916\1606725143.py:223: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  product_option = product_option.groupby("product_no", group_keys=False).apply(_calc_max_buy_qty)  # 적용


완료: 피프틴양식_ver2.xlsx 상품통합_ver2.xlsx


# 언어별 카테고리 추가

In [8]:
import pandas as pd  # 데이터 처리
from pathlib import Path  # 경로 처리
import math  # 올림 계산

# =========================
# 0) 파일 경로
# =========================
SRC_XLSX = Path("상품DB_251218.xlsx")  # 원본 엑셀 경로
CAT_XLSX = Path("피프틴양식_작성중.xlsx")  # 카테고리 마스터 엑셀 경로
OUT_7TABLES = Path("피프틴양식_ver2.xlsx")  # 7개 테이블 출력
OUT_MERGED = Path("상품통합_ver2.xlsx")  # 통합DB 출력

# =========================
# 1) 고정 마스터 (language)
# =========================
LANGUAGE_ROWS = [  # 언어 마스터 고정값
    ("01", "한국어", "ko"),  # 한국어
    ("02", "영어", "en"),  # 영어
    ("03", "태국어", "th"),  # 태국어
    ("04", "베트남어", "vi"),  # 베트남어
    ("05", "인도네시아어", "id"),  # 인도네시아어
]  # 언어 목록 끝

# =========================
# 2) 옵션 단위 코드
# =========================
UNIT_CODE = {  # 옵션단위 코드
    "EA": "01",  # 낱개
    "PACK": "02",  # 팩
    "BUNDLE": "03",  # 번들
    "BOX": "04",  # 박스
}  # 옵션단위 끝

# =========================
# 3) 유틸 함수
# =========================
def zfill_num(s: pd.Series, n: int) -> pd.Series:  # 0채우기
    return s.astype("Int64").astype("string").str.zfill(n)  # 숫자->문자->zfill

def safe_col(df: pd.DataFrame, name: str) -> pd.Series:  # 컬럼 없으면 빈값
    return df[name] if name in df.columns else pd.Series([pd.NA] * len(df))  # 안전 반환

def rename_cols(df: pd.DataFrame, col_map: dict) -> pd.DataFrame:  # 컬럼명 바꾸기
    return df.rename(columns=col_map)  # 컬럼명 변경

# =========================
# 4) 원본 시트 읽기
# =========================
base = pd.read_excel(SRC_XLSX, sheet_name=0)  # 1시트(기본)
price = pd.read_excel(SRC_XLSX, sheet_name=1)  # 2시트(가격/다국어)

# =========================
# 4-1) 카테고리 마스터 읽기(상위/하위)
# =========================
top_master = pd.read_excel(CAT_XLSX, sheet_name=1, dtype="string")  # 2번째 시트(상위)
sub_master = pd.read_excel(CAT_XLSX, sheet_name=2, dtype="string")  # 3번째 시트(하위)

top_master = top_master.rename(columns={  # 상위 컬럼명 통일
    "카테고리 No (PK)": "top_category_no",  # PK
    "언어 No (FK)": "language_no",  # 언어
    "카테고리명": "category_name",  # 이름
})  # 끝

sub_master = sub_master.rename(columns={  # 하위 컬럼명 통일
    "하위 카테고리 No (PK)": "sub_category_no",  # PK
    "상위 카테고리 No (FK)": "top_category_no",  # 상위FK
    "언어 No (FK)": "language_no",  # 언어
    "하위 카테고리명": "sub_category_name",  # 이름
})  # 끝

top_master["top_category_no"] = top_master["top_category_no"].astype("string").str.zfill(4)  # 4자리
top_master["language_no"] = top_master["language_no"].astype("string").str.zfill(2)  # 2자리
sub_master["sub_category_no"] = sub_master["sub_category_no"].astype("string").str.zfill(7)  # 7자리
sub_master["top_category_no"] = sub_master["top_category_no"].astype("string").str.zfill(4)  # 4자리
sub_master["language_no"] = sub_master["language_no"].astype("string").str.zfill(2)  # 2자리

# =========================
# 4-2) KO 하위카테고리명 -> 번호 매핑 만들기
# =========================
sub_map_ko = (  # 매핑DF
    sub_master[sub_master["language_no"] == "01"]  # 한국어만
    .set_index("sub_category_name")[["sub_category_no", "top_category_no"]]  # 이름키
    .to_dict("index")  # dict
)  # 끝

# =========================
# 4-3) 품목코드 정리 + product_no 생성
# =========================
base["품목코드"] = base["품목코드"].astype("Int64")  # 품목코드 정리
price["품목코드"] = price["품목코드"].astype("Int64")  # 품목코드 정리

base["product_no"] = zfill_num(base["품목코드"], 8)  # 상품No(8자리)
price["product_no"] = zfill_num(price["품목코드"], 8)  # 상품No(8자리)

# =========================
# 5) “처음부터 상품명 붙여서” 조합용 기준 DF 만들기
# =========================
base_cols = ["product_no"]  # 기본 컬럼
for col in ["품목명", "국가명", "과세/면세", "카테고리명", "단위", "규격정보"]:  # 필요한 컬럼들
    if col in base.columns:  # 있으면
        base_cols.append(col)  # 컬럼 추가

df = price.merge(  # 가격시트에
    base[base_cols].copy(),  # 기본정보 붙이기
    on="product_no",  # 상품No로 조인
    how="left",  # 왼쪽 유지
)  # merge 끝

df["origin_product_name"] = safe_col(df, "품목명_x")  # 원본 상품명

# =========================
# 6) 카테고리 번호 생성 (KO 마스터 참조)
# =========================
df["sub_name_ko"] = safe_col(df, "카테고리").fillna(df["카테고리명"])  # 하위카테고리명 후보
df["sub_category_no"] = df["sub_name_ko"].map(lambda x: sub_map_ko.get(x, {}).get("sub_category_no"))  # 하위번호
df["top_category_no"] = df["sub_name_ko"].map(lambda x: sub_map_ko.get(x, {}).get("top_category_no"))  # 상위번호

df["top_category_no_ko"] = df["top_category_no"]  # KO 상위카테고리 번호 백업
df["sub_category_no_ko"] = df["sub_category_no"]  # KO 하위카테고리 번호 백업

# =========================
# 7) 7개 테이블 생성
# =========================
language = pd.DataFrame(LANGUAGE_ROWS, columns=["language_no", "language_name", "language_code"])  # 언어테이블

top_category = (  # 상위카테고리
    top_master[["top_category_no", "language_no", "category_name"]]  # 마스터에서 가져오기
    .dropna()  # 빈값 제거
    .drop_duplicates()  # 중복 제거
)  # 끝

sub_category = (  # 하위카테고리
    sub_master[["sub_category_no", "top_category_no", "language_no", "sub_category_name"]]  # 마스터에서 가져오기
    .dropna()  # 빈값 제거
    .drop_duplicates()  # 중복 제거
)  # 끝

product = pd.DataFrame()  # product 만들기
product["product_no"] = df["product_no"]  # 상품No
product["top_category_no"] = df["top_category_no"]  # 상위FK(4자리, KO기준)
product["sub_category_no"] = df["sub_category_no"]  # 하위FK(7자리, KO기준)
product["one_time_limit_yn"] = 0  # 기본 0
product["tax_free_yn"] = (df["과세/면세"] == "면세").astype("int")  # 면세면 1
product["wms_code"] = zfill_num(df["품목코드"], 8)  # WMS코드
product["item_code"] = zfill_num(df["품목코드"], 8)  # 품목코드
product["coupang_proxy_yn"] = 0  # 기본 0
product["safety_stock"] = pd.NA  # 안전재고
product["temp_soldout_yn"] = 0  # 임시품절
product["timesale_start_dt"] = pd.NA  # 타임세일 시작
product["timesale_end_dt"] = pd.NA  # 타임세일 종료
product["timesale_discount_type"] = pd.NA  # 할인유형
product["timesale_discount_value"] = pd.NA  # 할인값
product = product.drop_duplicates(subset=["product_no"])  # 중복 제거

def make_product_lang(lang_no: str, name_col: str) -> pd.DataFrame:  # 언어별 상품정보
    t = pd.DataFrame()  # 임시 DF
    t["product_no"] = df["product_no"]  # FK
    t["language_no"] = lang_no  # 언어 FK
    if name_col == "품목명":  # 원본명
        t["product_name"] = df["origin_product_name"]  # 한글명
    else:  # 다국어명
        t["product_name"] = safe_col(df, name_col)  # 해당명
    t["origin"] = safe_col(df, "국가명")  # 원산지
    t["description"] = pd.NA  # 상세설명
    t["notice"] = pd.NA  # 고시
    t = t[t["product_name"].notna()]  # 이름없는 행 제거
    t["product_lang_no"] = t["product_no"] + t["language_no"]  # PK
    return t[["product_lang_no", "product_no", "language_no", "product_name", "origin", "description", "notice"]]  # 반환

pl_ko = make_product_lang("01", "품목명")  # 한국
pl_en = make_product_lang("02", "상품 영어명")  # 영어
pl_th = make_product_lang("03", "태국")  # 태국
pl_vi = make_product_lang("04", "베트남")  # 베트남
pl_id = make_product_lang("05", "인도네시아")  # 인도네시아
product_lang = pd.concat([pl_ko, pl_en, pl_th, pl_vi, pl_id], ignore_index=True)  # 합치기

# =========================
# 7-6) product_option
# =========================
opt_rows = []  # 옵션 행
for _, r in df.iterrows():  # 한줄씩
    pno = r["product_no"]  # 상품No
    base_unit = str(r["단위"]) if pd.notna(r.get("단위")) else "EA"  # 기본단위
    base_code = UNIT_CODE.get(base_unit, "01")  # 코드
    opt_rows.append((pno + base_code, pno, base_code, r.get("규격정보"), 1, pd.NA, pd.NA))  # 기본옵션
    if pd.notna(r.get("팩품목코드")):  # 팩
        opt_rows.append((pno + "02", pno, "02", r.get("규격정보"), r.get("팩품목환산수량\n(1팩에 몇 개)"), pd.NA, pd.NA))  # 팩옵션
    if pd.notna(r.get("번들품목코드")):  # 번들
        opt_rows.append((pno + "03", pno, "03", r.get("규격정보"), r.get("번들품목환산수량\n(번들에 몇 개)"), pd.NA, pd.NA))  # 번들옵션
    if pd.notna(r.get("박스품목 환산수량")):  # 박스
        opt_rows.append((pno + "04", pno, "04", r.get("규격정보"), r.get("박스품목 환산수량"), pd.NA, pd.NA))  # 박스옵션

product_option = pd.DataFrame(  # 옵션 DF
    opt_rows,  # 데이터
    columns=["product_option_no", "product_no", "option_unit", "weight", "sale_unit", "max_buy_qty", "temp_soldout_text"],  # 컬럼
)  # DF 생성
product_option = product_option.drop_duplicates(subset=["product_option_no"])  # 중복 제거

# =========================
# 7-6-1) 최대 구매 수량 자동 계산(바로 위 단위 기준)
# =========================
product_option["option_unit"] = product_option["option_unit"].astype("string").str.zfill(2)  # 단위코드 정리
product_option["sale_unit"] = pd.to_numeric(product_option["sale_unit"], errors="coerce")  # 판매단위 숫자화

def _calc_max_buy_qty(g: pd.DataFrame) -> pd.DataFrame:  # 계산
    g = g.copy()  # 복사
    g["unit_int"] = pd.to_numeric(g["option_unit"], errors="coerce")  # 숫자화
    g["max_buy_qty"] = pd.NA  # 초기화
    units = sorted([u for u in g["unit_int"].dropna().unique().tolist()])  # 정렬
    next_map = {}  # 매핑
    for u in units:  # 반복
        higher = [x for x in units if x > u]  # 상위 후보
        next_map[u] = min(higher) if higher else None  # 바로 위
    for u in units:  # 반복
        nu = next_map.get(u)  # 위 단위
        if nu is None:  # 없으면
            continue  # 종료
        cur_sale = g.loc[g["unit_int"] == u, "sale_unit"]  # 현재
        next_sale = g.loc[g["unit_int"] == nu, "sale_unit"]  # 위
        if cur_sale.empty or next_sale.empty:  # 없으면
            continue  # 종료
        cur_val = cur_sale.iloc[0]  # 값
        next_val = next_sale.iloc[0]  # 값
        if pd.isna(cur_val) or pd.isna(next_val) or cur_val == 0:  # 예외
            continue  # 종료
        qty = int(math.ceil(float(next_val) / float(cur_val)))  # 올림
        g.loc[g["unit_int"] == u, "max_buy_qty"] = qty  # 저장
    g = g.drop(columns=["unit_int"])  # 정리
    return g  # 반환

product_option = product_option.groupby("product_no", group_keys=False).apply(_calc_max_buy_qty)  # 적용

# =========================
# 7-7) product_option_lang_price (언어별 가격 “전부”)
# =========================
def add_price_row(rows: list, opt_no: str, lang_no: str, inbound, cost, sale, sale_unit):  # 가격행 추가
    rows.append((opt_no + lang_no, lang_no, opt_no, inbound, cost, sale, sale_unit))  # 행 추가

price_rows = []  # 가격 행
for _, r in df.iterrows():  # 반복
    pno = r["product_no"]  # 상품No
    base_unit = str(r["단위"]) if pd.notna(r.get("단위")) else "EA"  # 기본단위
    base_code = UNIT_CODE.get(base_unit, "01")  # 코드
    ea_no = pno + base_code  # 옵션No

    ea_sale_unit = 1  # EA 판매단위
    for lang_no in ["01", "02", "03", "04", "05"]:  # 언어 반복
        add_price_row(price_rows, ea_no, lang_no, r.get("입고단가"), r.get("낱개단가"), r.get("낱개단가"), ea_sale_unit)  # 추가

    if pd.notna(r.get("팩품목코드")):  # 팩이면
        pack_no = pno + "02"  # 팩 옵션No
        pack_sale_unit = r.get("팩품목환산수량\n(1팩에 몇 개)")  # 팩 판매단위
        for lang_no in ["01", "02", "03", "04", "05"]:  # 언어 반복
            add_price_row(price_rows, pack_no, lang_no, r.get("입고단가"), r.get("팩단가"), r.get("팩단가"), pack_sale_unit)  # 추가

    if pd.notna(r.get("번들품목코드")):  # 번들이면
        bun_no = pno + "03"  # 번들 옵션No
        bun_sale_unit = r.get("번들품목환산수량\n(번들에 몇 개)")  # 번들 판매단위
        for lang_no in ["01", "02", "03", "04", "05"]:  # 언어 반복
            add_price_row(price_rows, bun_no, lang_no, r.get("입고단가"), r.get("번들단가"), r.get("번들단가"), bun_sale_unit)  # 추가

    if pd.notna(r.get("박스품목 환산수량")):  # 박스면
        box_no = pno + "04"  # 박스 옵션No
        box_sale_unit = r.get("박스품목 환산수량")  # 박스 판매단위
        for lang_no in ["01", "02", "03", "04", "05"]:  # 언어 반복
            add_price_row(price_rows, box_no, lang_no, r.get("입고단가"), r.get("박스단가"), r.get("박스단가"), box_sale_unit)  # 추가

product_option_lang_price = pd.DataFrame(  # 가격 DF
    price_rows,  # 데이터
    columns=["option_lang_no", "language_no", "product_option_no", "inbound_price", "cost_price", "sale_price", "sale_unit"],  # 컬럼
)  # DF 생성

product_option_lang_price["sale_unit"] = pd.to_numeric(product_option_lang_price["sale_unit"], errors="coerce").fillna(1)  # 판매단위 정리
product_option_lang_price["inbound_price"] = pd.to_numeric(product_option_lang_price["inbound_price"], errors="coerce")  # 숫자화
product_option_lang_price["cost_price"] = pd.to_numeric(product_option_lang_price["cost_price"], errors="coerce")  # 숫자화
product_option_lang_price["sale_price"] = pd.to_numeric(product_option_lang_price["sale_price"], errors="coerce")  # 숫자화

product_option_lang_price["inbound_price"] = product_option_lang_price["inbound_price"] * product_option_lang_price["sale_unit"]  # 입고가*판매단위
product_option_lang_price["cost_price"] = product_option_lang_price["cost_price"] * product_option_lang_price["sale_unit"]  # 원가*판매단위
product_option_lang_price["sale_price"] = product_option_lang_price["sale_price"] * product_option_lang_price["sale_unit"]  # 판매가*판매단위

product_option_lang_price = product_option_lang_price.drop(columns=["sale_unit"])  # 임시 판매단위 제거
product_option_lang_price = product_option_lang_price.drop_duplicates(subset=["product_option_no", "language_no"])  # 중복 제거

# =========================
# 8) 통합DB 생성
# =========================
merged = product_option_lang_price.merge(product_option, on="product_option_no", how="left")  # 가격+옵션
merged = merged.merge(product, on="product_no", how="left")  # +상품
merged = merged.merge(product_lang, on=["product_no", "language_no"], how="left")  # +언어별 상품
merged = merged.merge(df[["product_no", "origin_product_name"]].drop_duplicates(), on="product_no", how="left")  # +원본명

# =========================
# 8-0) (핵심) 통합DB에서 언어별 카테고리번호로 변환
# =========================
merged["top_category_no"] = merged["top_category_no"].astype("string")  # 상위번호 문자로 통일
merged["sub_category_no"] = merged["sub_category_no"].astype("string")  # 하위번호 문자로 통일
merged["language_no"] = merged["language_no"].astype("string").str.zfill(2)  # 언어번호 2자리 보정

merged["top_category_no"] = merged["top_category_no"].where(  # NA면 그대로 두기
    merged["top_category_no"].isna(),  # 상위번호 NA 체크
    merged["language_no"] + merged["top_category_no"].str[2:],  # 앞2자리만 언어로 교체
)  # 상위번호 언어별 변환

merged["sub_category_no"] = merged["sub_category_no"].where(  # NA면 그대로 두기
    merged["sub_category_no"].isna(),  # 하위번호 NA 체크
    merged["language_no"] + merged["sub_category_no"].str[2:],  # 앞2자리만 언어로 교체
)  # 하위번호 언어별 변환

# =========================
# 8-1) 컬럼명 한글로 변경(여기서 적용)
# =========================
language = rename_cols(language, {  # language 컬럼명 변경
    "language_no": "언어 No",  # 언어 PK
    "language_name": "언어명",  # 언어명
    "language_code": "언어 코드",  # 언어코드
})  # language 끝

top_category = rename_cols(top_category, {  # top_category 컬럼명 변경
    "top_category_no": "카테고리 No (PK)",  # 카테고리 PK
    "language_no": "언어 No (FK)",  # 언어 FK
    "category_name": "카테고리명",  # 카테고리명
})  # top_category 끝

sub_category = rename_cols(sub_category, {  # sub_category 컬럼명 변경
    "sub_category_no": "하위 카테고리 No (PK)",  # 하위 PK
    "top_category_no": "상위 카테고리 No (FK)",  # 상위 FK
    "language_no": "언어 No (FK)",  # 언어 FK
    "sub_category_name": "하위 카테고리명",  # 하위명
})  # sub_category 끝

product = rename_cols(product, {  # product 컬럼명 변경
    "product_no": "상품 No (PK)",  # 상품 PK
    "top_category_no": "상위 카테고리 No (FK)",  # 상위 FK
    "sub_category_no": "하위 카테고리 No (FK)",  # 하위 FK
    "one_time_limit_yn": "1회 구매 제한 상품 여부 (0-X, 1-O)",  # 제한 여부
    "tax_free_yn": "면세 상품 여부 (0-X, 1-O)",  # 면세 여부
    "wms_code": "WMS 코드",  # WMS 코드
    "item_code": "품목 코드",  # 품목 코드
    "coupang_proxy_yn": "쿠팡 대리구매 상품 여부 (0-X, 1-O)",  # 쿠팡 여부
    "safety_stock": "안전재고",  # 안전재고
    "temp_soldout_yn": "임시품절 여부 (0-X, 1-O)",  # 임시품절
    "timesale_start_dt": "타임세일 시작일시",  # 타임세일 시작
    "timesale_end_dt": "타임세일 종료일시",  # 타임세일 종료
    "timesale_discount_type": "타임세일 할인 유형 (percent => 정률, price => 정액)",  # 할인유형
    "timesale_discount_value": "타임세일 할인값",  # 할인값
})  # product 끝

product_lang = rename_cols(product_lang, {  # product_lang 컬럼명 변경
    "product_lang_no": "언어별 상품정보 No (PK)",  # 언어별 PK
    "sub_category_no": "하위 카테고리 No (FK)",  # 하위 FK
    "product_no": "상품 No (FK)",  # 상품 FK
    "language_no": "언어 No (FK)",  # 언어 FK
    "product_name": "상품명",  # 상품명
    "origin": "원산지",  # 원산지
    "description": "상세 설명",  # 상세설명
    "notice": "상품 정보고시",  # 고시
})  # product_lang 끝

product_option = rename_cols(product_option, {  # product_option 컬럼명 변경
    "product_option_no": "상품 옵션 No (PK)",  # 옵션 PK
    "product_no": "상품 No (FK)",  # 상품 FK
    "option_unit": "옵션 단위(EA, BUNDLE, PACK, BOX)",  # 옵션단위
    "weight": "무게",  # 무게
    "sale_unit": "판매 단위",  # 판매단위
    "max_buy_qty": "최대 구매 수량",  # 최대수량
    "temp_soldout_text": "임시 품절여부",  # 임시품절 텍스트
})  # product_option 끝

product_option_lang_price = rename_cols(product_option_lang_price, {  # 가격 컬럼명 변경
    "option_lang_no": "언어별 상품 옵션정보 No (PK)",  # PK
    "language_no": "언어 No (FK)",  # 언어 FK
    "product_option_no": "상품 옵션 No (FK)",  # 옵션 FK
    "inbound_price": "입고가",  # 입고가
    "cost_price": "원가",  # 원가
    "sale_price": "판매가",  # 판매가
})  # 가격 끝

merged = rename_cols(merged, {  # merged 컬럼명 변경
    "origin_product_name": "원본 상품명",  # 원본명
    "option_lang_no": "언어별 상품 옵션정보 No (PK)",  # PK
    "language_no": "언어 No (FK)",  # 언어 FK
    "product_option_no": "상품 옵션 No (FK)",  # 옵션 FK
    "product_no": "상품 No (FK)",  # 상품 FK
    "inbound_price": "입고가",  # 입고가
    "cost_price": "원가",  # 원가
    "sale_price": "판매가",  # 판매가
    "option_unit": "옵션 단위(EA, BUNDLE, PACK, BOX)",  # 옵션단위
    "weight": "무게",  # 무게
    "sale_unit": "판매 단위",  # 판매단위
    "max_buy_qty": "최대 구매 수량",  # 최대수량
    "temp_soldout_text": "임시 품절여부",  # 임시품절 텍스트
    "product_name": "상품명",  # 상품명
    "origin": "원산지",  # 원산지
    "description": "상세 설명",  # 상세설명
    "notice": "상품 정보고시",  # 고시
    "top_category_no": "상위 카테고리 No (FK)",  # 상위 FK
    "sub_category_no": "하위 카테고리 No (FK)",  # 하위 FK
    "one_time_limit_yn": "1회 구매 제한 상품 여부 (0-X, 1-O)",  # 제한 여부
    "tax_free_yn": "면세 상품 여부 (0-X, 1-O)",  # 면세 여부
    "wms_code": "WMS 코드",  # WMS 코드
    "item_code": "품목 코드",  # 품목 코드
    "coupang_proxy_yn": "쿠팡 대리구매 상품 여부 (0-X, 1-O)",  # 쿠팡 여부
    "safety_stock": "안전재고",  # 안전재고
    "temp_soldout_yn": "임시품절 여부 (0-X, 1-O)",  # 임시품절
    "timesale_start_dt": "타임세일 시작일시",  # 타임세일 시작
    "timesale_end_dt": "타임세일 종료일시",  # 타임세일 종료
    "timesale_discount_type": "타임세일 할인 유형 (percent => 정률, price => 정액)",  # 할인유형
    "timesale_discount_value": "타임세일 할인값",  # 할인값
})  # merged 끝

# =========================
# 9) 엑셀 저장 (7테이블 / 통합DB)
# =========================
with pd.ExcelWriter(OUT_7TABLES, engine="openpyxl") as w:  # 7테이블 저장
    language.to_excel(w, sheet_name="언어 등록 양식", index=False)  # language
    top_category.to_excel(w, sheet_name="상위 카테고리 등록 양식", index=False)  # top_category
    sub_category.to_excel(w, sheet_name="하위 카테고리 등록 양식", index=False)  # sub_category
    product.to_excel(w, sheet_name="상품 등록 양식", index=False)  # product
    product_lang.to_excel(w, sheet_name="언어별 상품 정보 등록 양식", index=False)  # product_lang
    product_option.to_excel(w, sheet_name="상품 옵션 등록 양식", index=False)  # product_option
    product_option_lang_price.to_excel(w, sheet_name="언어별 상품 옵션정보 등록 양식", index=False)  # price

with pd.ExcelWriter(OUT_MERGED, engine="openpyxl") as w:  # 통합 저장
    merged.to_excel(w, sheet_name="merged_all", index=False)  # merged

print("완료:", OUT_7TABLES, OUT_MERGED)  # 출력


C:\Users\jusun\AppData\Local\Temp\ipykernel_13752\2350407171.py:231: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  product_option = product_option.groupby("product_no", group_keys=False).apply(_calc_max_buy_qty)  # 적용


완료: 피프틴양식_ver2.xlsx 상품통합_ver2.xlsx
